<a href="https://colab.research.google.com/github/pyrenaaaaaa/Emotion-Aware-Companion/blob/text/sentiment140.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Change to your project directory
%cd /content/drive/MyDrive/EmotionCompanion/

Mounted at /content/drive
/content/drive/MyDrive/EmotionCompanion


In [2]:
# Move to your project directory
%cd /content/drive/MyDrive/EmotionCompanion/

# Delete the Git repository (permanently removes Git from this project)
!rm -rf .git

/content/drive/MyDrive/EmotionCompanion


# Define Variables

In [3]:
# Define Key Variables
num_epochs = 3
batch_size = 16
learning_rate = 3e-5
patience = 2  # Early stopping patience
step_size = 1  # LR scheduler step size
gamma = 0.9  # LR scheduler decay factor
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# print(f"Variables Set. Training on: {device}")

In [4]:
# Install required libaries
!pip install emoji transformers datasets torch tensorflow pandas scikit-learn -q
!pip install nlpaug textattack accelerate peft evaluate -q
!pip install nlpaug textattack -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Imports

In [5]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
import re
import emoji
import matplotlib.pyplot as plt
import seaborn as sns
import nlpaug.augmenter.word as naw
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification, AdamW
from transformers import AutoModelForSequenceClassification
import tensorflow as tf
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.char as nac
import random
import warnings
warnings.filterwarnings("ignore")
from torch.optim.lr_scheduler import StepLR
import torch.nn as nn
import torch.optim as optim

# Define Paths

In [6]:
base_path = "/content/drive/MyDrive/EmotionCompanion"
models_path = os.path.join(base_path, "models")
os.makedirs(models_path, exist_ok=True)

sentiment140_path = "/content/drive/MyDrive/EmotionCompanion/datasets/text/sentiment140.csv"

# Load Sentiment140 Dataset

In [7]:
sentiment140_data = pd.read_csv(sentiment140_path, encoding="latin-1", header=None)
sentiment140_data.columns = ["polarity", "id", "date", "query", "user", "text"]
sentiment140_data = sentiment140_data.drop(["id", "date", "query", "user"], axis=1)
sentiment140_data.rename(columns={"polarity": "label"}, inplace=True)

# Select 30K Positive & Negative Samples
NUM_SAMPLES = 30000
negative_samples = sentiment140_data[sentiment140_data["label"] == 0][:NUM_SAMPLES]
positive_samples = sentiment140_data[sentiment140_data["label"] == 4][:NUM_SAMPLES]
positive_samples["label"] = 1  # Convert label 4 → 1

sentiment140_data = pd.concat([negative_samples, positive_samples]).reset_index(drop=True)
sentiment140_data["label"] = sentiment140_data["label"].map({0: "Negative", 1: "Positive"})

# Data Preprocessing

In [8]:
sentiment140_data['old_length'] = sentiment140_data['text'].apply(len)

# Regex Patterns
REPEATED_CHAR_PATTERN = re.compile(r'(.)\1{2,}')
EMOJIS = sorted(emoji.EMOJI_DATA, key=len, reverse=True)
MULTI_EMOJI_PATTERN = re.compile(r"({})\1+".format("|".join(map(re.escape, EMOJIS))))

emoticon_dict = {
    ':)': '<smile_face>', ':(': '<sad_face>', ':D': '<big_smile>', '<3': '<heart>', '¯_(ツ)_/¯': '<shrug>',
}

def preprocess_text(text):
    text = MULTI_EMOJI_PATTERN.sub(r"\1", text)
    text = REPEATED_CHAR_PATTERN.sub(r'\1\1\1', text)
    text = re.sub(r'u/\w+', '<user>', text)
    text = re.sub(r'\d+', '<NUM>', text)
    for emoticon, name in emoticon_dict.items():
        text = text.replace(emoticon, name)
    return re.sub(r'\s+', ' ', text).strip()

sentiment140_data["cleaned_text"] = sentiment140_data["text"].apply(preprocess_text)
sentiment140_data['new_length'] = sentiment140_data['cleaned_text'].apply(len)

# Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(sentiment140_data["cleaned_text"], sentiment140_data["label"], test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

# Data Augmentation

In [10]:
# Install required libraries
!pip install --upgrade nlpaug nltk

# Import and force re-download missing NLTK resources
import nltk

nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')  # This fixes the specific error
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [11]:
# Synonym Replacement (SR)
syn_aug = naw.SynonymAug(aug_src='wordnet', aug_max=2)

# Random Deletion (RD)
def random_deletion(text, p=0.2):
    if isinstance(text, list):
        text = " ".join(text)  # Convert list to string if needed

    words = text.split()
    if len(words) == 1:
        return text  # If only one word, don't remove
    return " ".join([word for word in words if random.uniform(0, 1) > p])

# Apply Data Augmentation to Training Data
def augment_text(text):
    if isinstance(text, list):
        text = " ".join(text)  # Convert list to string if needed

    if random.random() < 0.5:
        text = syn_aug.augment(text)  # Synonym Replacement
    if random.random() < 0.5:
        text = random_deletion(text)  # Random Deletion
    return text

# Ensure X_train is a Pandas Series
X_train_series = pd.Series(X_train)

# Augment Training Data
X_train_aug = X_train_series.apply(augment_text)

# Combine Original + Augmented Data
X_train_final = pd.concat([X_train_series, X_train_aug], ignore_index=True)
y_train_final = pd.concat([pd.Series(y_train), pd.Series(y_train)], ignore_index=True)

print(f"Data Augmentation Completed! New training size: {len(X_train_final)}")

Data Augmentation Completed! New training size: 86400


# Load BERTweet Model & Tokenizer

In [12]:
# Load BERTweet Tokenizer
tokenizer_sent = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)

# Function to Ensure Text Data is Valid Before Tokenization
def clean_text_data(texts):
    if isinstance(texts, pd.Series):
        texts = texts.dropna().astype(str).tolist()  # Convert to list & ensure all are strings
    elif isinstance(texts, list):
        texts = [str(x) if isinstance(x, (str, int, float)) else "" for x in texts]  # Convert non-strings
        texts = [x for x in texts if x.strip()]  # Remove empty strings
    else:
        raise ValueError(f"Unexpected data type {type(texts)}")

    return texts

# Function to Tokenize Text
def tokenize_texts(tokenizer, texts, max_length=128):
    texts = clean_text_data(texts)  # Ensure valid text input
    return tokenizer(
        texts, max_length=max_length, padding="max_length", truncation=True, return_tensors="pt"
    )  # "pt" for PyTorch tensors

# Clean & Prepare Data for Tokenization
X_train_final = clean_text_data(X_train_final)
X_val = clean_text_data(X_val)
X_test = clean_text_data(X_test)

# Tokenize Data
print("🔄 Tokenizing Sentiment140 Dataset...")
X_train_tokens = tokenize_texts(tokenizer_sent, X_train_final)
X_val_tokens = tokenize_texts(tokenizer_sent, X_val)
X_test_tokens = tokenize_texts(tokenizer_sent, X_test)

# Convert Labels to PyTorch Tensors
y_train_tensor = torch.tensor(pd.get_dummies(y_train_final).values, dtype=torch.float32)
y_val_tensor = torch.tensor(pd.get_dummies(y_val).values, dtype=torch.float32)
y_test_tensor = torch.tensor(pd.get_dummies(y_test).values, dtype=torch.float32)

print("Tokenization & Label Conversion Successful!")

config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/843k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.91M [00:00<?, ?B/s]

🔄 Tokenizing Sentiment140 Dataset...
Tokenization & Label Conversion Successful!


In [13]:
from torch.utils.data import TensorDataset, DataLoader

# Step 1: Ensure X_train_tokens and y_train_tensor have the same length
min_length = min(len(X_train_tokens["input_ids"]), len(y_train_tensor))

# Trim to match lengths
X_train_tokens_trimmed = {key: value[:min_length] for key, value in X_train_tokens.items()}
y_train_tensor_trimmed = y_train_tensor[:min_length]

# Do the same for validation & test sets
min_length_val = min(len(X_val_tokens["input_ids"]), len(y_val_tensor))
X_val_tokens_trimmed = {key: value[:min_length_val] for key, value in X_val_tokens.items()}
y_val_tensor_trimmed = y_val_tensor[:min_length_val]

min_length_test = min(len(X_test_tokens["input_ids"]), len(y_test_tensor))
X_test_tokens_trimmed = {key: value[:min_length_test] for key, value in X_test_tokens.items()}
y_test_tensor_trimmed = y_test_tensor[:min_length_test]

print(f"Fixed Sizes - Train: {X_train_tokens_trimmed['input_ids'].shape}, {y_train_tensor_trimmed.shape}")
print(f"Fixed Sizes - Validation: {X_val_tokens_trimmed['input_ids'].shape}, {y_val_tensor_trimmed.shape}")
print(f"Fixed Sizes - Test: {X_test_tokens_trimmed['input_ids'].shape}, {y_test_tensor_trimmed.shape}")

# Step 2: Create DataLoader function
def create_dataloader(X_tokens, y_labels, batch_size=16):
    dataset = TensorDataset(X_tokens["input_ids"], X_tokens["attention_mask"], y_labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Step 3: Create DataLoaders with Fixed Data
train_loader = create_dataloader(X_train_tokens_trimmed, y_train_tensor_trimmed, batch_size)
val_loader = create_dataloader(X_val_tokens_trimmed, y_val_tensor_trimmed, batch_size)
test_loader = create_dataloader(X_test_tokens_trimmed, y_test_tensor_trimmed, batch_size)

print(f"DataLoaders Created! Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")


Fixed Sizes - Train: torch.Size([86368, 128]), torch.Size([86368, 2])
Fixed Sizes - Validation: torch.Size([4800, 128]), torch.Size([4800, 2])
Fixed Sizes - Test: torch.Size([12000, 128]), torch.Size([12000, 2])
DataLoaders Created! Train: 5398, Val: 300, Test: 750


# Define Optimizer & Loss

In [14]:
# Load BERTweet Model & Tokenizer for Sentiment140
tokenizer_sent = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)
model_sentiment = AutoModelForSequenceClassification.from_pretrained("vinai/bertweet-base", num_labels=2)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_sentiment.to(device)

print("Sentiment140 model loaded and moved to:", device)

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Sentiment140 model loaded and moved to: cuda


In [15]:
# Define Optimizer & Loss Function
optimizer = optim.AdamW(model_sentiment.parameters(), lr=3e-5)
loss_fn = nn.BCEWithLogitsLoss()

# Train Model

In [ ]:
# Clear GPU Memory Before Training
torch.cuda.empty_cache()

# Initialize Loss & Accuracy Tracking
train_loss_history = []
train_acc_history = []
val_loss_history = []
val_acc_history = []

num_epochs = 3
batch_size = 16
learning_rate = 3e-5
# Learning Rate Scheduler
scheduler = StepLR(optimizer, step_size=1, gamma=0.9)  # Reduce LR by 10% per epoch

best_val_loss = float("inf")  # Track the best validation loss
patience = 2  # Number of epochs to wait before stopping
counter = 0  # Counter to track worsening epochs

for epoch in range(num_epochs):
    model_sentiment.train()
    total_loss = 0
    correct_train = 0
    total_train = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Training]", leave=True)

    for batch in train_bar:
        input_ids, attention_mask, labels = [x.to(device) for x in batch]

        optimizer.zero_grad()
        outputs = model_sentiment(input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = (torch.sigmoid(outputs.logits) > 0.5).float()
        correct_train += (preds == labels).sum().item()
        total_train += labels.numel()

        train_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    avg_train_loss = total_loss / len(train_loader)
    train_accuracy = correct_train / total_train
    train_loss_history.append(avg_train_loss)
    train_acc_history.append(train_accuracy)

    model_sentiment.eval()
    val_loss = 0
    correct_val = 0
    total_val = 0

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Validation]", leave=True)

    with torch.no_grad():
        for batch in val_bar:
            input_ids, attention_mask, labels = [x.to(device) for x in batch]
            outputs = model_sentiment(input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            val_loss += loss.item()
            preds = (torch.sigmoid(outputs.logits) > 0.5).float()
            correct_val += (preds == labels).sum().item()
            total_val += labels.numel()

            val_bar.set_postfix({"Val Loss": f"{loss.item():.4f}"})

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct_val / total_val
    val_loss_history.append(avg_val_loss)
    val_acc_history.append(val_accuracy)

    print(f"\n📊 Epoch {epoch+1} Summary: Train Loss = {avg_train_loss:.4f}, Train Acc = {train_accuracy:.4f}, Val Loss = {avg_val_loss:.4f}, Val Acc = {val_accuracy:.4f}")

    scheduler.step()  # Apply learning rate decay

    # EARLY STOPPING LOGIC
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0  # Reset counter if validation loss improves
    else:
        counter += 1  # Increase counter if validation loss worsens

        if counter >= patience:
            print(f"\n Early stopping triggered after {epoch+1} epochs.")
            break  # Stop training

# Save Model After Training
torch.save(model_sentiment.state_dict(), "sentiment140_model.pth")
print("Model saved successfully as 'sentiment140_model.pth'!")

Epoch 1/3 [Training]:   0%|          | 0/5398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Epoch 1/3 [Validation]: 100%|██████████| 300/300 [00:08<00:00, 36.88it/s, Val Loss=0.7127]



📊 Epoch 1 Summary: Train Loss = 0.6501, Train Acc = 0.5906, Val Loss = 0.6940, Val Acc = 0.5106


Epoch 2/3 [Training]:  15%|█▍        | 802/5398 [01:07<06:27, 11.86it/s, Loss=0.6812]

# Evaluate Model

In [ ]:
# Move model to evaluation mode
model_sentiment.eval()

# Collect Predictions & True Labels
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [x.to(device) for x in batch]

        outputs = model_sentiment(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Convert logits to probabilities and apply threshold
        preds = torch.sigmoid(logits) > 0.5

        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

# Convert lists to numpy arrays
y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)

# Compute Accuracy & F1 Scores
accuracy = accuracy_score(y_true, y_pred)
f1_micro = f1_score(y_true, y_pred, average="micro")
f1_macro = f1_score(y_true, y_pred, average="macro")

print(f"\n **Final Evaluation Results for Sentiment140**")
print(f" Accuracy: {accuracy:.4f}")
print(f" F1-Score (Micro): {f1_micro:.4f}")
print(f" F1-Score (Macro): {f1_macro:.4f}\n")

# Classification Report
print("\n📊 **Classification Report:**")
print(classification_report(y_true, y_pred, target_names=["Negative", "Positive"], digits=4))

# Confusion Matrix
plt.figure(figsize=(6, 4))
cm = confusion_matrix(y_true.argmax(axis=1), y_pred.argmax(axis=1))
sns.heatmap(cm, annot=True, cmap="Blues", fmt="d", xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
plt.title("Confusion Matrix - Sentiment140")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Training & Validation Loss Plot
plt.figure(figsize=(10, 5))
plt.plot(train_loss_history, label="Train Loss")
plt.plot(val_loss_history, label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.show()

# Training & Validation Accuracy Plot
plt.figure(figsize=(10, 5))
plt.plot(train_acc_history, label="Train Accuracy")
plt.plot(val_acc_history, label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training & Validation Accuracy")
plt.legend()
plt.show()

print("\n **Training Complete & Final Results Saved!** 🚀")

In [ ]:
torch.save(model_sentiment.state_dict(), "/content/drive/MyDrive/EmotionCompanion/models/sentiment_model.pth")
print("Sentiment model saved successfully!")